In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import matplotlib
import matplotlib.ticker as ticker
matplotlib.rcParams["figure.figsize"] = (12, 6)

import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize
from tqdm import tqdm

plt.style.use('ggplot')

from scipy.stats import norm
from scipy.stats import binom

from datetime import datetime

import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px

from math import exp, factorial


pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [30]:
from pathlib import Path

# Absolute path of THIS notebook file
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


current = NOTEBOOK_DIR
while current.name != "data_processing":
    current = current.parent

DATA_DIR = current / "data"

print("Notebook dir:", NOTEBOOK_DIR)
print("Data processing dir:", current)
print("Data dir:", DATA_DIR)
print("Exists:", DATA_DIR.exists())
print("Files:", [p.name for p in DATA_DIR.iterdir()])

Notebook dir: c:\Evailable Projects\evercharge-research\data_processing\analysis\notebooks
Data processing dir: c:\Evailable Projects\evercharge-research\data_processing
Data dir: c:\Evailable Projects\evercharge-research\data_processing\data
Exists: True
Files: ['22BZ3340B_abnormal_reasons.csv', '23BZ0762E_short_sessions.csv', '23BZ4143A_short_sessions.csv', '23BZ4145A_zero_sessions.csv', '23BZ4233E_short_sessions.csv', '23BZ4237E_short_sessions.csv', '2nd_CZ_UFC_CB98000080_authorization_data.csv', '3rd_CZ_UFC_CB98000079_authorization_data.csv', '4th_23BZ0536E_authorization_data.csv', '5th_23BZ4228E_authorization_data.csv', '6th_23BZ0497B_authorization_data.csv', 'auth_transaction_data.csv', 'charging_sessions_edri.csv', 'charging_stations_overview_latest_edri.csv', 'edri_authorization_local_2025.csv', 'edri_authorization_remote_2025.csv', 'edri_start_transaction_2025.csv', 'erdi_local_remote_authorization_2025.csv', 'local_vs_remote_sessions_edri_jan_2025.csv', 'results.csv', 'top_st

In [31]:

df = pd.read_csv(DATA_DIR / "results.csv")

In [32]:
df["date"] = pd.to_datetime(df["date"])

In [33]:
tournament_mapping = {
    "Friendly": "friendly",
    "FIFA World Cup": "worldcup",
    "AFC Asian Cup qualification": "qualifier",
    "AFF Championship qualification": "qualifier",
    "CONCACAF Nations League qualification": "qualifier",
    "African Cup of Nations qualification": "qualifier",
    "UEFA Euro qualification": "qualifier",
    "FIFA World Cup qualification": "qualifier",
    "Arab Cup qualification": "qualifier",
    "Gold Cup qualification": "qualifier",
    "Oceania Nations Cup qualification": "qualifier",
    "Copa América qualification": "qualifier",
    "CONIFA World Football Cup qualification": "qualifier",
    "EAFF Championship qualification": "qualifier",
    "ASEAN Championship qualification": "qualifier",
    "AFC Asian Cup": "continental",
    "African Cup of Nations": "continental",
    "UEFA Euro": "continental",
    "Copa América": "continental",
    "Gold Cup": "continental",
    "Oceania Nations Cup": "continental",
    "UEFA Nations League": "continental",
    "CONCACAF Nations League": "continental",
    "AFF Championship": "continental",
    "ASEAN Championship": "continental",
    "EAFF Championship": "continental",
    "SAFF Cup": "continental",
    "WAFF Championship": "continental",
    "CAFA Nations Cup": "continental",
    "Gulf Cup": "continental",
    "COSAFA Cup": "continental",
    "Baltic Cup": "continental",
    "Pacific Games": "continental",
    "Island Games": "continental",
    "Indian Ocean Island Games": "continental",
    "CONIFA World Football Cup": "continental",
    "CONIFA European Football Cup": "continental",
    "CONIFA Africa Football Cup": "continental",
    "CONIFA South America Football Cup": "continental",
    "CONIFA Asia Cup": "continental"
}

In [34]:
df["match_type"] = df["tournament"].map(tournament_mapping)
df = df[df["match_type"].notna()]

In [35]:
# -----------------------------
# LAST 2 YEARS DATA
# -----------------------------
end_date = df["date"].max()
start_date = end_date - pd.DateOffset(years=2)
df = df[df["date"] >= start_date].reset_index(drop=True)

In [36]:
# TIME DECAY
# -----------------------------
def compute_time_weights(match_dates, current_date, halfperiod_days=90):

    delta = (current_date - match_dates).dt.days
    return 0.5 ** (delta / halfperiod_days)


In [37]:
# IMPORTANCE WEIGHTS
# -----------------------------
def importance_weight(match_types, weights_dict):

    return match_types.map(weights_dict).values


In [38]:
# NEGATIVE LOG LIKELIHOOD
# -----------------------------
def neg_log_likelihood(params, home_idx, away_idx,
                       scores_h, scores_a,
                       match_types, match_dates,
                       current_date, weights_dict):

    n_teams = len(params) - 2

    r = params[:n_teams]
    h = params[n_teams]
    d = params[n_teams + 1]

    delta = r[home_idx] + h - r[away_idx]

    PH = 1 - norm.cdf((d - delta) / np.sqrt(2))
    PA = norm.cdf((-d - delta) / np.sqrt(2))
    PD = 1 - PH - PA

    PH = np.clip(PH, 1e-10, 1)
    PA = np.clip(PA, 1e-10, 1)
    PD = np.clip(PD, 1e-10, 1)

    outcome_prob = np.where(scores_h > scores_a, PH,
                    np.where(scores_h < scores_a, PA, PD))

    w_time = compute_time_weights(match_dates, current_date)
    w_imp = importance_weight(match_types, weights_dict)

    w = w_time * w_imp

    return -np.sum(w * np.log(outcome_prob))

In [39]:
# RPS (CORRECT)
# -----------------------------
def compute_rps(pred, obs):

    cdf_pred = np.cumsum(pred, axis=1)
    cdf_obs = np.cumsum(obs, axis=1)

    return np.mean(np.sum((cdf_pred - cdf_obs) ** 2, axis=1) / 2)

In [40]:
# ROLLING VALIDATION
# -----------------------------
def rolling_validation(df, weights_dict):

    rps_list = []

    months = pd.date_range(
        df["date"].min(),
        df["date"].max() - pd.DateOffset(months=1),
        freq="MS"
    )

    for current_start in tqdm(months):

        train = df[df["date"] < current_start]
        test = df[
            (df["date"] >= current_start) &
            (df["date"] < current_start + pd.DateOffset(months=1))
        ]

        if len(train) == 0 or len(test) == 0:
            continue

        teams = np.unique(np.concatenate([train.home_team, train.away_team]))
        team_idx = {t: i for i, t in enumerate(teams)}

        home_idx = train.home_team.map(team_idx).values
        away_idx = train.away_team.map(team_idx).values

        scores_h = train.home_score.values
        scores_a = train.away_score.values

        match_types = train.match_type
        match_dates = train.date

        n_teams = len(teams)

        x0 = np.concatenate([np.zeros(n_teams), [0.1, 0.1]])

        res = minimize(
            neg_log_likelihood,
            x0,
            args=(
                home_idx,
                away_idx,
                scores_h,
                scores_a,
                match_types,
                match_dates,
                train.date.max(),
                weights_dict
            ),
            method="BFGS"
        )

        r = res.x[:n_teams]
        h = res.x[n_teams]
        d = res.x[n_teams + 1]

        test_home = test.home_team.map(team_idx)
        test_away = test.away_team.map(team_idx)

        mask = test_home.notna() & test_away.notna()

        test_home = test_home[mask].astype(int).values
        test_away = test_away[mask].astype(int).values

        scores_h = test.home_score[mask].values
        scores_a = test.away_score[mask].values

        delta = r[test_home] + h - r[test_away]

        PH = 1 - norm.cdf((d - delta) / np.sqrt(2))
        PA = norm.cdf((-d - delta) / np.sqrt(2))
        PD = 1 - PH - PA

        pred = np.stack([PH, PD, PA], axis=1)

        obs = np.zeros((len(scores_h), 3))

        for i in range(len(scores_h)):

            if scores_h[i] > scores_a[i]:
                obs[i] = [1, 0, 0]

            elif scores_h[i] < scores_a[i]:
                obs[i] = [0, 0, 1]

            else:
                obs[i] = [0, 1, 0]

        rps = compute_rps(pred, obs)

        rps_list.append(rps)

    return np.mean(rps_list)

In [41]:
# GRID SEARCH
# -----------------------------
weights_grid = [
    {"friendly": f, "qualifier": q, "continental": c, "worldcup": w}
    for f in [0.5, 1, 1.5]
    for q in [1, 2, 2.5]
    for c in [2.5, 3, 3.5]
    for w in [3.5, 4, 4.5]
]

best_rps = np.inf
best_weights = None

for w in tqdm(weights_grid):

    score = rolling_validation(df, w)

    print("Weights:", w, "RPS:", score)

    if score < best_rps:
        best_rps = score
        best_weights = w

print("Best Weights:", best_weights)
print("Best RPS:", best_rps)

  1%|          | 1/81 [17:14<22:59:48, 1034.85s/it]

Weights: {'friendly': 0.5, 'qualifier': 1, 'continental': 2.5, 'worldcup': 3.5} RPS: 0.2507297925087166


  2%|▏         | 2/81 [58:36<41:22:45, 1885.65s/it]

Weights: {'friendly': 0.5, 'qualifier': 1, 'continental': 2.5, 'worldcup': 4} RPS: 0.2507297925087166


  4%|▎         | 3/81 [1:41:38<47:44:57, 2203.82s/it]

...

Weights: {'friendly': 1.5, 'qualifier': 2.5, 'continental': 3.5, 'worldcup': 3.5} RPS: 0.2727384739493507


 99%|█████████▉| 80/81 [23:36:41<13:09, 789.23s/it]

Weights: {'friendly': 1.5, 'qualifier': 2.5, 'continental': 3.5, 'worldcup': 4} RPS: 0.2727384739493507


100%|██████████| 81/81 [23:50:51<00:00, 1059.89s/it]

Weights: {'friendly': 1.5, 'qualifier': 2.5, 'continental': 3.5, 'worldcup': 4.5} RPS: 0.2727384739493507
Best Weights: {'friendly': 0.5, 'qualifier': 1, 'continental': 3, 'worldcup': 3.5}
Best RPS: 0.24684581987037796
